# The Model

This notebook explains the `EmbeddingClassifier` model, a simple two-layer neural network that:
1. Projects embeddings to a new space (optional dimensionality reduction)
2. Classifies them into classes


## Architecture

The model has two components:
1. A **projection layer** that maps the input embedding to a hidden space
2. A **classification head** that outputs a score per class

```
Input Embedding (1024 dims)
        |
   [Linear + ReLU]    <-- projection layer
        |
Hidden Embedding (hidden_dim)
        |
   [Linear]           <-- classification head
        |
Class Logits (num_classes)
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent.parent))
from core.utils.model import EmbeddingClassifier

model = EmbeddingClassifier(
    input_dim=1024,
    hidden_dim=512,
    num_classes=10
)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

c:\Users\benmc\BaseAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
core.active_learner - INFO - Pre-warming UMAP (triggering numba JIT compilation)...
core.active_learner - INFO - UMAP pre-warming completed in 5.301332473754883


EmbeddingClassifier(
  (projection): Linear(in_features=1024, out_features=512, bias=True)
  (classifier): Linear(in_features=512, out_features=10, bias=True)
)

Total parameters: 529,930


## Forward pass

The model takes a batch of embeddings and returns raw logits (one score per class). To get probabilities, apply softmax.

In [2]:
# Create a fake embedding
fake_embedding = torch.randn(1, 1024)

with torch.no_grad():
    logits = model(fake_embedding)
    probabilities = F.softmax(logits, dim=1)

print(f"Input shape:  {fake_embedding.shape}")
print(f"Output shape: {logits.shape}")
print(f"\nProbabilities sum to: {probabilities.sum().item():.4f}")
print(f"\nTop 3 class probabilities:")
top_probs, top_classes = torch.topk(probabilities, 3)
for i in range(3):
    print(f"  Class {top_classes[0, i].item()}: {top_probs[0, i].item():.4f}")

Input shape:  torch.Size([1, 1024])
Output shape: torch.Size([1, 10])

Probabilities sum to: 1.0000

Top 3 class probabilities:
  Class 3: 0.2734
  Class 2: 0.2589
  Class 4: 0.1673


## Training loop

Training follows a standard supervised learning loop: forward pass, compute loss, backpropagate, update weights.

In [3]:
# Fake training data
batch_size = 8
X_batch = torch.randn(batch_size, 1024)
y_batch = torch.randint(0, 10, (batch_size,))

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Single training step
model.train()
optimizer.zero_grad()
logits = model(X_batch)
loss = criterion(logits, y_batch)
loss.backward()
optimizer.step()

with torch.no_grad():
    predictions = torch.argmax(logits, dim=1)
    accuracy = (predictions == y_batch).float().mean()

print(f"Loss: {loss.item():.4f}")
print(f"Accuracy: {accuracy.item():.4f}")
print(f"\nPredictions: {predictions.tolist()}")
print(f"True labels:  {y_batch.tolist()}")

Loss: 2.2784
Accuracy: 0.2500

Predictions: [1, 1, 4, 4, 4, 4, 4, 4]
True labels:  [0, 0, 3, 4, 0, 4, 1, 2]


## Summary

The model does two things:
1. **Projects** input embeddings to a hidden space which is then used for visualisation
2. **Classifies** embeddings into classes

As more labelled data is added through active learning, the model improves at distinguishing between classes. You do not need to modify the model to use BaseAL. The `ActiveLearner` handles model creation, training, and inference for you.